# Prepare the 2016 London respondent extract

This notebook creates the analysis file used for the eight-year composite comparison. It keeps respondents whose `LA_2023` value is one of the 33 London local authorities, selects the requested geography, demographic, weight and activity variables, and adds a constant `year` column.

The notebook writes two files:

- `active_lives_year1_london_125_stable_composites.csv` ? respondent-level data
- `active_lives_year1_london_125_stable_composites_variables.csv` ? variable dictionary

The original SAV is read only. No SAV file is created.

## 1. Imports

In [1]:
from pathlib import Path

import pandas as pd
import pyreadstat
from openpyxl import load_workbook

## 2. File paths and naming rules

Run the notebook from the project folder. The activity prefixes below correspond to the four measures requested for each stable composite.

In [2]:
project_dir = Path.cwd()

source_sav = (
    project_dir
    / "spss"
    / "spss28"
    / "active_lives_survey_nov_15-16_data_year_1_shared_20250106.sav"
)
composite_workbook = project_dir / "eight_year_composites.xlsx"

data_file = project_dir / "active_lives_1516_london_125.csv"
variable_file = (
    project_dir / "active_lives_1516_london_125_variables.csv"
)

activity_prefixes = ("MEMS7_", "MEMS7GR_", "DAYS10P60GR_", "MONTHS_12_")

if not source_sav.exists():
    raise FileNotFoundError(source_sav)
if not composite_workbook.exists():
    raise FileNotFoundError(composite_workbook)

## 3. Read the list of stable composites

Only the 125 suffixes on the `Stable composites` sheet are used. The activity label is retained for the variable dictionary.

In [3]:
workbook = load_workbook(composite_workbook, read_only=True, data_only=True)
stable_sheet = workbook["Stable composites"]

stable_composites = [
    {"suffix": str(row[0]).strip(), "label": str(row[1]).strip()}
    for row in stable_sheet.iter_rows(min_row=2, values_only=True)
    if row[0]
]

if len(stable_composites) != 125:
    raise ValueError(
        f"Expected 125 stable composites, found {len(stable_composites)}"
    )

print(f"Stable composites: {len(stable_composites)}")

Stable composites: 125


## 4. Build the variable list

SPSS variable names are matched without regard to case, but the original spelling is kept in the output. All variables beginning with `wt_` are treated as weight variables.

In [4]:
_, sav_metadata = pyreadstat.read_sav(source_sav, metadataonly=True)
source_name = {name.lower(): name for name in sav_metadata.column_names}

core_spec = [
    ("Other", "serial"),
    ("Other", "mode"),
    ("Other", "Month"),
    ("Geography", "LA_2023"),
    ("Geography", "Reg9"),
    ("Geography", "LondInOut"),
    ("Demographics", "age16plus"),
    ("Demographics", "Age9"),
    ("Demographics", "Disab3"),
]
core_spec.extend(
    ("Demographics", f"disty{number}_POP") for number in range(1, 14)
)

core_variables = []
for category, requested_name in core_spec:
    matched_name = source_name.get(requested_name.lower())
    if matched_name is None:
        raise KeyError(f"Variable not found in the SAV: {requested_name}")
    core_variables.append({"category": category, "name": matched_name})

weight_variables = [
    name for name in sav_metadata.column_names if name.lower().startswith("wt_")
]

activity_variables = []
for composite in stable_composites:
    for prefix in activity_prefixes:
        requested_name = f"{prefix}{composite['suffix']}"
        matched_name = source_name.get(requested_name.lower())
        activity_variables.append(
            {
                "name": matched_name or requested_name,
                "suffix": composite["suffix"],
                "label": composite["label"],
                "in_source": matched_name is not None,
            }
        )

missing_activity_variables = [
    item["name"] for item in activity_variables if not item["in_source"]
]

print(f"Core variables: {len(core_variables)}")
print(f"Weight variables: {len(weight_variables)}")
print(f"Activity variables: {len(activity_variables)}")
print("Not present in the Year 1 SAV:", missing_activity_variables)

Core variables: 22
Weight variables: 8
Activity variables: 500
Not present in the Year 1 SAV: ['MEMS7_HULAHOOP_P27', 'MEMS7GR_HULAHOOP_P27', 'DAYS10P60GR_HULAHOOP_P27', 'MONTHS_12_HULAHOOP_P27']


## 5. Read the selected source columns

User-defined SPSS missing values are kept as their original numeric codes. This matters for variables such as `LondInOut`, where zero is stored as a user-missing value in the source file.

In [5]:
source_columns = (
    [item["name"] for item in core_variables]
    + weight_variables
    + [item["name"] for item in activity_variables if item["in_source"]]
)

source_data, selected_metadata = pyreadstat.read_sav(
    source_sav,
    usecols=source_columns,
    apply_value_formats=False,
    formats_as_category=False,
    user_missing=True,
)

print(f"Source respondents: {len(source_data):,}")
print(f"Source columns read: {len(source_data.columns):,}")

Source respondents: 198,911
Source columns read: 526


## 6. Keep London respondents

London local authority labels begin with the ONS code prefix `E09`. This gives 33 London codes, including the City of London.

In [6]:
la_labels = selected_metadata.variable_value_labels["LA_2023"]
london_codes = {
    value for value, label in la_labels.items() if str(label).startswith("E09")
}

if len(london_codes) != 33:
    raise ValueError(f"Expected 33 London LA codes, found {len(london_codes)}")

london_data = source_data.loc[source_data["LA_2023"].isin(london_codes)].copy()

print(f"London respondents retained: {len(london_data):,}")

London respondents retained: 19,887


## 7. Add the year and arrange the columns

`year` is inserted after `serial`. Four Hula hoop variables are listed in the comparison workbook but are absent from the Year 1 SAV, so they are included as blank columns.

In [7]:
for variable_name in missing_activity_variables:
    london_data[variable_name] = pd.NA

ordered_columns = (
    [item["name"] for item in core_variables]
    + weight_variables
    + [item["name"] for item in activity_variables]
)
london_data = london_data.loc[:, ordered_columns]
london_data.insert(1, "year", 2016)

print("First columns:", list(london_data.columns[:6]))

First columns: ['serial', 'year', 'mode', 'Month', 'LA_2023', 'Reg9']


## 8. Create the variable dictionary

The dictionary follows the exact column order of the data file and records whether each field came from the source SAV or was added during preparation.

In [8]:
dictionary_rows = []

for item in core_variables:
    dictionary_rows.append(
        {
            "category": item["category"],
            "variable": item["name"],
            "activity_suffix": "",
            "activity_label": "",
            "source_status": "source variable",
        }
    )

dictionary_rows.insert(
    1,
    {
        "category": "Other",
        "variable": "year",
        "activity_suffix": "",
        "activity_label": "",
        "source_status": "constant value: 2016",
    },
)

for variable_name in weight_variables:
    dictionary_rows.append(
        {
            "category": "Weight",
            "variable": variable_name,
            "activity_suffix": "",
            "activity_label": "",
            "source_status": "source variable",
        }
    )

for item in activity_variables:
    dictionary_rows.append(
        {
            "category": "Activity",
            "variable": item["name"],
            "activity_suffix": item["suffix"],
            "activity_label": item["label"],
            "source_status": (
                "source variable" if item["in_source"] else "blank placeholder"
            ),
        }
    )

variable_dictionary = pd.DataFrame(dictionary_rows)

## 9. Validate the extract before writing files

These checks stop the notebook if the filter, year field, activity groups or variable dictionary do not match the intended structure.

In [9]:
assert london_data.columns[1] == "year"
assert london_data["year"].eq(2016).all()
assert london_data["LA_2023"].isin(london_codes).all()
assert london_data["serial"].notna().all()
assert london_data["serial"].is_unique
assert variable_dictionary["variable"].tolist() == london_data.columns.tolist()

for prefix in activity_prefixes:
    prefix_count = sum(name.startswith(prefix) for name in london_data.columns)
    assert prefix_count == 125, f"{prefix}: expected 125 columns, found {prefix_count}"

assert london_data[missing_activity_variables].isna().all().all()

print(f"Validated respondents: {len(london_data):,}")
print(f"Validated columns: {len(london_data.columns):,}")
print(f"Inner London: {(london_data['LondInOut'] == 1).sum():,}")
print(f"Outer London: {(london_data['LondInOut'] == 2).sum():,}")

Validated respondents: 19,887
Validated columns: 531
Inner London: 7,796
Outer London: 12,091


## 10. Write the two CSV files

In [10]:
london_data.to_csv(data_file, index=False, encoding="utf-8-sig")
variable_dictionary.to_csv(variable_file, index=False, encoding="utf-8-sig")

print(f"Data file: {data_file}")
print(f"Variable file: {variable_file}")

Data file: y:\final\UKDA-8223-spss\active_lives_1516_london_125.csv
Variable file: y:\final\UKDA-8223-spss\active_lives_1516_london_125_variables.csv


## 11. Check the written files

The final check reads the headers and the two key filter fields back from disk.

In [11]:
written_data = pd.read_csv(
    data_file, usecols=["serial", "year", "LA_2023", "LondInOut"]
)
written_dictionary = pd.read_csv(variable_file)

assert len(written_data) == len(london_data)
assert written_data["year"].eq(2016).all()
assert written_data["LA_2023"].isin(london_codes).all()
assert len(written_dictionary) == len(london_data.columns)

print(f"Final data shape: {len(written_data):,} rows x {len(london_data.columns):,} columns")
print(f"Variable dictionary rows: {len(written_dictionary):,}")

Final data shape: 19,887 rows x 531 columns
Variable dictionary rows: 531
